# bind_tools와 병렬 호출

요청, 실행, ToolMessage를 구분합니다.

## 준비

In [1]:
# 상위 폴더에서 환경 파일과 공개 주문 CSV를 찾습니다.
from pathlib import Path
from dotenv import load_dotenv

def find_up(name: str) -> Path | None:
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        candidate = folder / name
        if candidate.exists():
            return candidate
    return None

load_dotenv(find_up(".env"))
data_path = find_up("data/orders.csv")
assert data_path is not None, "공개 묶음의 챕터 폴더 또는 notebooks 폴더에서 실행합니다."
print("Python 실습 준비 완료")


Python 실습 준비 완료


In [2]:
# 같은 모델을 도구 요청과 최종 답변에 사용합니다.
import os
from langchain.chat_models import init_chat_model
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
MODEL = "gpt-5.4-mini"
model = init_chat_model(MODEL, model_provider="openai", temperature=0)
print({"model": MODEL, "temperature": 0, "예상_절별_비용_USD_상한": 1.0})


{'model': 'gpt-5.4-mini', 'temperature': 0, '예상_절별_비용_USD_상한': 1.0}


In [3]:
# 주문 CSV를 읽는 함수를 준비합니다.
from pathlib import Path
from typing import Any

import pandas as pd


def fetch_orders(csv_path: str | Path, *, status: str | None = None) -> list[dict[str, Any]]:
    """CSV 주문을 읽고 선택한 상태의 행을 사전 목록으로 반환합니다."""
    orders = pd.read_csv(csv_path, dtype={"order_id": "string", "customer_id": "string"})
    if status is not None:
        orders = orders.loc[orders["status"] == status]
    return orders.to_dict(orient="records")


In [4]:
# 상태로 목록을 조회하는 도구를 준비합니다.
from langchain.tools import tool
@tool
def list_orders_by_status(status: str) -> list[dict[str, Any]]:
    """주문 상태(paid, pending, cancelled)로 주문 목록을 조회합니다."""
    return fetch_orders(data_path, status=status)


In [5]:
# 주문 ID로 한 건을 조회하는 도구를 준비합니다.
@tool
def get_order_by_id(order_id: str) -> dict:
    """정확한 주문 ID로 주문 한 건의 상태와 금액을 조회합니다."""
    rows = fetch_orders(data_path)
    return next((r for r in rows if r["order_id"] == order_id), {"error": "주문을 찾지 못했습니다."})

@tool
def summarize_order_amounts(status: str) -> dict:
    """주문 상태별 건수와 총 주문 금액을 집계합니다."""
    rows = fetch_orders(data_path, status=status)
    return {"status": status, "count": len(rows), "total": sum(r["amount"] for r in rows)}


## 병렬 호출 요청

In [6]:
# 한 응답에 두 주문의 호출 요청이 있는지 확인합니다.
bound = model.bind_tools([get_order_by_id], parallel_tool_calls=True)
question = "O101과 O103 주문을 각각 조회해 줘. 각 주문마다 도구를 한 번씩 호출해."
ai_message = bound.invoke(question)
print({"content": ai_message.text, "tool_calls": ai_message.tool_calls, "usage": ai_message.usage_metadata})
assert len(ai_message.tool_calls) == 2
assert {call["args"]["order_id"] for call in ai_message.tool_calls} == {"O101", "O103"}


{'content': '', 'tool_calls': [{'name': 'get_order_by_id', 'args': {'order_id': 'O101'}, 'id': 'call_x8JKo8joJe1bqXdvT0fL8FLG', 'type': 'tool_call'}, {'name': 'get_order_by_id', 'args': {'order_id': 'O103'}, 'id': 'call_wNneZQoLq4pEQAXH0cx2Ta47', 'type': 'tool_call'}], 'usage': {'input_tokens': 163, 'output_tokens': 55, 'total_tokens': 218, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}


## 실제 실행

In [7]:
# 각 호출 요청을 순차 실행하여 ToolMessage를 얻습니다.
tool_messages = [get_order_by_id.invoke(call) for call in ai_message.tool_calls]
for message in tool_messages:
    print({"type": type(message).__name__, "tool_call_id": message.tool_call_id, "content": message.content})


{'type': 'ToolMessage', 'tool_call_id': 'call_x8JKo8joJe1bqXdvT0fL8FLG', 'content': '{"order_id": "O101", "customer_id": "C001", "status": "paid", "amount": 32000}'}
{'type': 'ToolMessage', 'tool_call_id': 'call_wNneZQoLq4pEQAXH0cx2Ta47', 'content': '{"order_id": "O103", "customer_id": "C001", "status": "paid", "amount": 27000}'}


## ID 대응

In [8]:
# 요청 ID와 결과 ID가 같은 순서로 연결되는지 검사합니다.
requested_ids = [call["id"] for call in ai_message.tool_calls]
returned_ids = [message.tool_call_id for message in tool_messages]
print({"requested": requested_ids, "returned": returned_ids, "same": requested_ids == returned_ids})
assert requested_ids == returned_ids and len(set(requested_ids)) == 2


{'requested': ['call_x8JKo8joJe1bqXdvT0fL8FLG', 'call_wNneZQoLq4pEQAXH0cx2Ta47'], 'returned': ['call_x8JKo8joJe1bqXdvT0fL8FLG', 'call_wNneZQoLq4pEQAXH0cx2Ta47'], 'same': True}


## 결과 전달

In [9]:
# 도구 결과를 모델에 전달하고 사용량 기준 추정 비용을 계산합니다.
final = bound.invoke([{"role": "user", "content": question}, ai_message, *tool_messages])
print(final.text)
print(final.usage_metadata)
assert final.text and not final.tool_calls
assert ai_message.usage_metadata is not None and final.usage_metadata is not None
usages = [ai_message.usage_metadata, final.usage_metadata]
estimated_cost_usd = sum(u["input_tokens"] * 0.75 + u["output_tokens"] * 4.5 for u in usages) / 1_000_000
assert estimated_cost_usd < 1
print({"estimated_cost_usd": estimated_cost_usd})


조회 결과입니다.

- O101: 상태 paid, 금액 32,000원
- O103: 상태 paid, 금액 27,000원
{'input_tokens': 290, 'output_tokens': 36, 'total_tokens': 326, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
{'estimated_cost_usd': 0.00074925}


## 실험 결과 해석

호출 두 개는 확인했지만 실행 시간의 동시성은 측정하지 않았습니다.